In [1]:
import pandas as pd

filepath = '/Users/Mach/dev/aps/data/2026_Dmodel_data/master_dataset_car.parquet'
df = pd.read_parquet(filepath)

# Display the first few rows
df.head()

,vin_date,vin,zip,py,companyid,reference_num,policyid,poleffdt,poleffdt_imps,polexpdt_raw,...,territory_core3,territory_otc,territory_pd,territory_pip,A1_Score,B2_score,C1_Collision_Score,C1_Comprehensive_Score,C1_First_Party_Liability_Score,C1_Third_Party_Liability_Score
0,KMHWF35H45A178516_04-26-2022,KMHWF35H45A178516,43613,2022,C172621,C172621,H246865,2022-04-26,2022-04-26,2023-04-26,...,OH1,OH4,OH1,OH0,479.0,391.0,665.0,795.0,221.0,569.0
1,KMHWF35H75A128502_11-21-2020,KMHWF35H75A128502,78758,2020,C117026,C117026,30623212,2020-11-21,2020-11-21,2021-05-21,...,TX3,TX5,TX3,TX1,479.0,353.0,625.0,755.0,229.0,574.0
2,KMHD35LE5DU068088_05-03-2020,KMHD35LE5DU068088,92315,2020,C117026,C117026,22150816,2020-05-03,2020-05-03,2021-05-03,...,CA1,CA2,CA2,CA0,483.0,553.0,637.0,752.0,499.0,611.0
3,KMHD35LE5DU047158_05-30-2021,KMHD35LE5DU047158,30519,2021,C147884,C147884,91111025KG,2021-05-30,2021-05-30,2021-11-30,...,GA2,GA3,GA2,GA0,483.0,338.0,637.0,752.0,499.0,611.0
4,KMHD35LE5DU082332_11-17-2019,KMHD35LE5DU082332,8075,2019,C090317,C090317,HPA00001212784,2019-11-17,2019-11-17,2020-05-17,...,NJ5,NJ5,NJ4,NJ4,483.0,495.0,647.0,756.0,491.0,605.0


In [ ]:
# Create a CSV of all column names
column_names = pd.DataFrame(df.columns, columns=['column_name'])
column_names.to_csv('column_names.csv', index=False)

print(f'Saved {len(df.columns)} column names to column_names.csv')
column_names

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_histogram(df, feature_name, bins=50, figsize=(10, 6), ax=None):
    """
    Create a histogram for a specified feature in the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the data
    feature_name : str
        The name of the column to create histogram for
    bins : int, optional
        Number of bins for the histogram (default: 50)
    figsize : tuple, optional
        Figure size as (width, height) in inches (default: (10, 6))
    ax : matplotlib.axes.Axes, optional
        Axes object to plot on. If None, creates new figure.
    
    Returns:
    --------
    fig, ax : tuple
        The figure and axes objects (fig is None if ax was provided)
    """
    if feature_name not in df.columns:
        raise ValueError(f"Feature '{feature_name}' not found in dataframe")
    
    data = df[feature_name].dropna()
    
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = None
    
    ax.hist(data, bins=bins, edgecolor='black', alpha=0.7)
    ax.set_xlabel(feature_name)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Histogram of {feature_name}')
    
    # Add statistics annotation
    stats_text = f'n={len(data):,}\nmean={data.mean():.2f}\nstd={data.std():.2f}\nmin={data.min():.2f}\nmax={data.max():.2f}'
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    if fig is not None:
        plt.tight_layout()
    
    return fig, ax

In [ ]:
def plot_categorical_distribution(df, feature_name, figsize=(10, 6), ax=None, max_categories=30):
    """
    Create a bar chart for a categorical feature in the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the data
    feature_name : str
        The name of the column to create bar chart for
    figsize : tuple, optional
        Figure size as (width, height) in inches (default: (10, 6))
    ax : matplotlib.axes.Axes, optional
        Axes object to plot on. If None, creates new figure.
    max_categories : int, optional
        Maximum number of categories to show (default: 30)
    
    Returns:
    --------
    fig, ax : tuple
        The figure and axes objects (fig is None if ax was provided)
    """
    if feature_name not in df.columns:
        raise ValueError(f"Feature '{feature_name}' not found in dataframe")
    
    data = df[feature_name].dropna()
    value_counts = data.value_counts()
    
    # Limit categories if too many
    if len(value_counts) > max_categories:
        value_counts = value_counts.head(max_categories)
        truncated = True
    else:
        truncated = False
    
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = None
    
    bars = ax.bar(range(len(value_counts)), value_counts.values, edgecolor='black', alpha=0.7)
    ax.set_xticks(range(len(value_counts)))
    ax.set_xticklabels([str(x)[:15] for x in value_counts.index], rotation=45, ha='right')
    ax.set_xlabel(feature_name)
    ax.set_ylabel('Count')
    
    title = f'Distribution of {feature_name}'
    if truncated:
        title += f' (top {max_categories} categories)'
    ax.set_title(title)
    
    # Add statistics annotation
    n_unique = df[feature_name].nunique()
    n_missing = df[feature_name].isna().sum()
    stats_text = f'n={len(data):,}\nunique={n_unique:,}\nmissing={n_missing:,}'
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    if fig is not None:
        plt.tight_layout()
    
    return fig, ax

In [ ]:
def is_categorical(series, max_unique_ratio=0.05, max_unique_count=20):
    """
    Determine if a pandas Series is categorical.
    
    Parameters:
    -----------
    series : pd.Series
        The series to check
    max_unique_ratio : float
        If unique values / total values < this ratio, consider categorical
    max_unique_count : int
        If unique values < this count, consider categorical
    
    Returns:
    --------
    bool : True if categorical, False if numerical
    """
    # Object and category dtypes are categorical
    if series.dtype == 'object' or series.dtype.name == 'category':
        return True
    
    # Boolean is categorical
    if series.dtype == 'bool':
        return True
    
    # Check if numeric column has few unique values (likely categorical)
    n_unique = series.nunique()
    n_total = len(series.dropna())
    
    if n_total == 0:
        return False
    
    unique_ratio = n_unique / n_total
    
    if n_unique <= max_unique_count or unique_ratio <= max_unique_ratio:
        return True
    
    return False

In [ ]:
# Create histogram for incloss_imps
fig, ax = plot_histogram(df, 'incloss_imps')
plt.show()

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm

def export_all_features_to_pdf(df, output_path='feature_distributions.pdf', figsize=(10, 6)):
    """
    Create histograms or bar charts for all features and save to PDF.
    One feature per page.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the data
    output_path : str
        Path to save the PDF file
    figsize : tuple
        Figure size for each page
    """
    with PdfPages(output_path) as pdf:
        for col in tqdm(df.columns, desc='Creating plots'):
            fig, ax = plt.subplots(figsize=figsize)
            
            try:
                if is_categorical(df[col]):
                    plot_categorical_distribution(df, col, ax=ax)
                else:
                    plot_histogram(df, col, ax=ax)
                
                plt.tight_layout()
                pdf.savefig(fig)
            except Exception as e:
                # Handle any errors (e.g., all NaN columns)
                ax.text(0.5, 0.5, f'Error plotting {col}:\n{str(e)}',
                       transform=ax.transAxes, ha='center', va='center',
                       fontsize=12, color='red')
                ax.set_title(f'{col} (Error)')
                pdf.savefig(fig)
            finally:
                plt.close(fig)
    
    print(f'Saved {len(df.columns)} feature plots to {output_path}')

In [ ]:
# Export all features to PDF (one feature per page)
export_all_features_to_pdf(df, 'feature_distributions.pdf')

In [ ]:
def get_feature_stats(df, feature_name):
    """
    Get basic statistics for a feature: min, max, and missing count.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe containing the data
    feature_name : str
        The name of the column to get statistics for
    
    Returns:
    --------
    dict : Dictionary containing min, max, and missing count
    """
    if feature_name not in df.columns:
        raise ValueError(f"Feature '{feature_name}' not found in dataframe")
    
    series = df[feature_name]
    
    stats = {
        'feature': feature_name,
        'min': series.min(),
        'max': series.max(),
        'missing_count': series.isna().sum(),
        'missing_pct': (series.isna().sum() / len(series)) * 100
    }
    
    print(f"Feature: {feature_name}")
    print(f"  Min: {stats['min']}")
    print(f"  Max: {stats['max']}")
    print(f"  Missing: {stats['missing_count']:,} ({stats['missing_pct']:.2f}%)")
    
    return stats

In [ ]:
# Get stats for dml_year_imps
stats = get_feature_stats(df, 'dml_year_imps')

In [ ]:
# Count records with -999.0 for dml_year_imps
count_999 = (df['dml_year_imps'] == -999.0).sum()
total_records = len(df)
pct_999 = (count_999 / total_records) * 100

print(f"Records with dml_year_imps == -999.0: {count_999:,} ({pct_999:.2f}%)")

Records with dml_year_imps == -999.0: 2,509 (0.01%)


: 